# Fisiere: text, CSV, imagini (predare)

Apasa **Copy to Drive** (sus, in stanga) inainte sa scrii ceva aici, altfel tot ce faci dispare cand inchizi tab-ul.

Ruleaza prima celula de mai jos, o singura data, ca sa aduci utilitarele cursului.

In [ ]:
import os, urllib.request
if not os.path.exists('onia.py'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/alexbirs27/iLoveBeer/main/cursuri/colab/onia.py', 'onia.py')
from onia import *
descarca('date/elevi.csv', 'date/09_forme.png', 'date/09_mic.png', 'anim_imagini.py')
from anim_imagini import *

## De ce fisiere

Pana acum datele veneau din `input()` sau din liste scrise de mana. La olimpiada (si in orice proiect de ML) datele stau in **fisiere**: un `train.csv` cu exemple, poze pentru un clasificator, iar raspunsul tau pleaca tot intr-un fisier, `submission.csv`. Cine nu stie sa citeasca si sa scrie fisiere nu poate nici macar sa-si trimita solutia.

Trei lucruri in acest notebook: **text**, **CSV** si **imagini**. Toate sunt, pana la urma, numere si siruri de caractere puse pe un disc.

Ca sa nu umplem folderul cu fisiere de test, scriem tot intr-un folder temporar. `ACASA` retine folderul cu datele cursului (`date/elevi.csv`, imaginile).

In [ ]:
import os, tempfile

ACASA = os.getcwd()                    # aici sunt date/elevi.csv si imaginile
os.chdir(tempfile.mkdtemp())           # de acum scriem intr-un folder temporar
print('lucram in', os.getcwd())

## 1. Fisiere text

Un fisier text e un sir lung de caractere, cu `\n` intre randuri. Ai nevoie de trei pasi: **deschizi**, **lucrezi**, **inchizi**. Instructiunea `with` inchide singura fisierul la sfarsit, chiar daca apare o eroare pe drum, deci o scrii mereu.

| mod | ce face |
|---|---|
| `'r'` | citeste (implicit); eroare daca fisierul nu exista |
| `'w'` | scrie; **sterge tot** ce era in fisier (il creeaza daca nu exista) |
| `'a'` | adauga la sfarsit, fara sa stearga |

**Prezice:** rulam de doua ori `open('x.txt', 'w')` si scriem de fiecare data `'a\n'`. Cate linii are fisierul?

<details><summary>Raspuns</summary>

Una singura. Modul `'w'` goleste fisierul cand il deschide, deci a doua rulare il ia de la zero. Pentru a adauga se foloseste `'a'`.
</details>

In [ ]:
with open('note.txt', 'w', encoding='utf-8') as f:
    f.write('Ana 9\n')                # write NU pune singur \n
    f.write('Dan 7\n')
    print('Eva 10', file=f)           # print scrie in fisier si pune el \n

with open('note.txt', 'a', encoding='utf-8') as f:
    f.write('Ion 8\n')                # 'a' = la sfarsit

with open('note.txt', encoding='utf-8') as f:
    print(f.read())

`print(..., file=f)` si `f.write(...)` fac aproape acelasi lucru. Diferenta: `print` pune singur `\n` si transforma orice in text, `write` primeste doar text si scrie exact ce ii dai.

Citirea. Cea mai folosita forma e bucla pe fisier: `for linie in f`. Fiecare `linie` vine **cu `\n` la capat**, de aceea urmeaza aproape mereu `.strip()` (taie spatiile si `\n` de la capete) sau `.split()` (taie dupa spatii).

In [ ]:
%%vezi
suma = 0
with open('note.txt', encoding='utf-8') as f:
    for linie in f:                       #: f se parcurge linie cu linie; linia are \n la capat
        nume, nota = linie.split()        #: split() fara argument taie dupa spatii si scapa si de \n
        suma += int(nota)

Celelalte metode de citire, ca sa le recunosti:

In [ ]:
with open('note.txt', encoding='utf-8') as f:
    tot = f.read()                    # tot fisierul, un singur string
print(repr(tot))

with open('note.txt', encoding='utf-8') as f:
    print(repr(f.readline()))         # o singura linie
    print(repr(f.readline()))         # urmatoarea: fisierul isi tine minte pozitia
    print(f.readlines())              # restul, ca lista de linii

with open('note.txt', encoding='utf-8') as f:
    print([l.strip() for l in f])     # cel mai des: bucla, cu strip

**De ce bucla si nu `read()`?** `read()` aduce tot fisierul in memorie deodata. La un fisier de cativa GB, programul moare. Bucla tine in memorie **o singura linie** la un moment dat. Masuram cu `tracemalloc` (memoria maxima ocupata) pe un fisier cu 300.000 de linii:

In [ ]:
import tracemalloc

with open('mare.txt', 'w') as f:
    for i in range(300_000):
        f.write(f'{i}\n')

def suma_cu_read():
    with open('mare.txt') as f:
        return sum(int(x) for x in f.read().split())    # tot fisierul in memorie

def suma_pe_linii():
    with open('mare.txt') as f:
        return sum(int(l) for l in f)                   # o linie odata

for fct in (suma_cu_read, suma_pe_linii):
    tracemalloc.start()
    rez = fct()
    _, varf = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f'{fct.__name__:15} suma={rez}  memorie maxima ~ {varf / 1e6:.2f} MB')

### Diacritice: `encoding`

Un fisier tine octeti, nu litere. `encoding` spune cum se transforma literele in octeti si invers. Standardul de azi e `'utf-8'`. Daca deschizi un fisier cu alt encoding decat cel in care a fost scris, obtii litere stricate sau eroare. Pe Windows `open` fara `encoding` foloseste altceva decat utf-8, deci codul care merge in Colab poate crapa pe calculatorul de acasa. **Scrie mereu `encoding='utf-8'`.**

Ca sa avem litere romanesti fara sa le tastam, le scriem cu escape-uri (`\u0103` este `a` cu semn, `\u0219` este `s` cu virgula):

In [ ]:
text = 'Ana are mere \u0219i pere, iar \u021b\u0103ranul are \u00eenv\u0103\u021b\u0103tor'
with open('ro.txt', 'w', encoding='utf-8') as f:
    f.write(text + '\n')

with open('ro.txt', encoding='utf-8') as f:
    print(f.read().strip())               # citit corect
with open('ro.txt', encoding='latin-1') as f:
    print(f.read().strip())               # citit cu alt encoding: litere stricate

### Cai si fisiere care nu exista

`open('note.txt')` cauta fisierul relativ la **folderul curent** al programului (`os.getcwd()`). `os.path.exists` iti spune daca exista, `os.path.join` lipeste cai corect pe orice sistem, iar o cale scrisa complet (absoluta) merge de oriunde.

In [ ]:
print(os.path.exists('note.txt'), os.path.exists('nu_exista.txt'))
print(sorted(os.listdir('.')))                    # ce fisiere sunt in folderul curent
cale_elevi = os.path.join(ACASA, 'date', 'elevi.csv')
print(os.path.exists(cale_elevi))

Un fisier lipsa nu trebuie sa-ti dea programul peste cap: prinzi `FileNotFoundError` cu `try/except`.

In [ ]:
%%vezi
def citeste(cale):
    try:
        with open(cale, encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return None                       #: nu exista: intoarcem None in loc sa crape programul

a = citeste('note.txt')
b = citeste('nu_exista.txt')

## 2. CSV

CSV (comma separated values) e un fisier text in care fiecare linie e un rand de tabel, iar valorile sunt separate prin virgula. De obicei prima linie e antetul (numele coloanelor). Aproape toate seturile de date de ML vin asa.

**Din topor**, un CSV se citeste cu ce stii deja: `split(',')`. Sa vedem unde se rupe. Fisierul de mai jos are trei feluri de linii: normala, cu o celula lipsa, si cu o virgula **in interiorul** unui nume:

In [ ]:
with open('mini.csv', 'w', encoding='utf-8') as f:
    f.write('nume,nota,clasa\n')
    f.write('Ana,9,10\n')
    f.write('Dan,,11\n')                     # nota lipseste
    f.write('"Popescu, Ion",8,12\n')         # virgula intre ghilimele face parte din nume

**Prezice:** cate campuri iese pentru fiecare linie, daca o tai cu `.split(',')`?

<details><summary>Raspuns</summary>

3, 3, 3 si... **4** pentru ultima. `split(',')` nu stie de ghilimele, taie si virgula din `"Popescu, Ion"`. Celula lipsa nu e o problema (da un sir gol, `''`), dar virgula din text strica tot.
</details>

In [ ]:
%%vezi
linii = ['Ana,9,10', 'Dan,,11', '"Popescu, Ion",8,12']
for linie in linii:
    campuri = linie.split(',')        #: taie la fiecare virgula, si la cea din ghilimele
    print(len(campuri), campuri)

Modulul `csv` din biblioteca standard stie regulile (ghilimele, virgule in text). `csv.reader` da fiecare rand ca lista de siruri. **Deschide fisierul cu `newline=''`**: modulul `csv` isi gestioneaza singur capetele de linie, iar fara asta pe Windows apar randuri goale intre randuri.

In [ ]:
import csv

with open('mini.csv', encoding='utf-8', newline='') as f:
    for rand in csv.reader(f):
        print(rand)

`csv.DictReader` foloseste prima linie ca nume de coloane si da fiecare rand ca dictionar. Codul se citeste mai bine (`r['nota']` in loc de `r[1]`) si nu se strica daca cineva schimba ordinea coloanelor.

**Atentie:** `csv` da mereu **siruri**. Celula lipsa e `''`, nu `0` si nu `None`. Conversia o faci tu, si trebuie sa prevezi celulele proaste:

In [ ]:
with open('mini.csv', encoding='utf-8', newline='') as f:
    for r in csv.DictReader(f):
        print(r['nume'], '->', repr(r['nota']))

In [ ]:
%%vezi
note = ['9', '', '8', 'absent']
bune = []
for t in note:
    try:
        bune.append(float(t))             #: float('') si float('absent') ridica ValueError
    except ValueError:
        pass                              #: celula proasta: o sarim
media = sum(bune) / len(bune)             #: media doar pe ce am putut citi

Acum pe un fisier real, `date/elevi.csv` (aproape 250 de elevi). Media notei la informatica, **de mana**, sarind peste celulele goale:

In [ ]:
suma, cate = 0, 0
with open(cale_elevi, encoding='utf-8', newline='') as f:
    for r in csv.DictReader(f):
        try:
            suma += float(r['nota_info'])
            cate += 1
        except ValueError:
            pass                          # celula goala
media_mana = suma / cate
print(cate, 'note valide; media =', round(media_mana, 4))

### Prima privire la pandas

Tot ce ai scris mai sus (deschis, `DictReader`, `try/except`, suma, numarat) e **o singura linie** in pandas. Il inveti pe larg in notebook-urile 10 si 11; aici vezi doar de ce il vei prefera:

In [ ]:
import pandas as pd

df = pd.read_csv(cale_elevi)
print(df.shape)
print(df['nota_info'].mean())             # mean() sare singur peste celulele lipsa (NaN)
verifica_egal('media de mana = pandas', media_mana, float(df['nota_info'].mean()))

Pandas ghiceste singur tipurile (coloana cu celule goale devine `float` cu `NaN`), sare peste valorile lipsa la statistici si e **mult mai rapid**: citirea e scrisa in C, nu in bucla de Python. Masuram pe un fisier cu 200.000 de randuri:

In [ ]:
import time
import numpy as np

x = np.random.default_rng(0).random(200_000).round(4)
with open('mare.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f)
    w.writerow(['id', 'x'])
    w.writerows(zip(range(len(x)), x))

def media_de_mana(cale):
    suma, n = 0, 0
    with open(cale, encoding='utf-8', newline='') as f:
        for r in csv.DictReader(f):
            suma += float(r['x'])
            n += 1
    return suma / n

t = time.perf_counter(); m1 = media_de_mana('mare.csv'); t1 = time.perf_counter() - t
t = time.perf_counter(); m2 = pd.read_csv('mare.csv')['x'].mean(); t2 = time.perf_counter() - t
print(f'de mana: {m1:.5f} in {t1:.3f}s | pandas: {m2:.5f} in {t2:.3f}s | ~{t1 / t2:.0f}x mai repede')

Scrierea unui CSV: `csv.writer` (cu `newline=''` si aici). Vezi ca pune singur ghilimele cand valoarea contine virgula. Asa nu mai scrii tu `','.join(...)`, care ar strica exact cazul de mai sus.

In [ ]:
with open('rezultate.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f)
    w.writerow(['nume', 'nota'])
    w.writerow(['Popescu, Ion', 9])       # virgula in text: writer pune ghilimele
    w.writerows([['Ana', 10], ['Dan', 7]])

with open('rezultate.csv', encoding='utf-8') as f:
    print(f.read())

## 3. Cum iti trimiti raspunsul: `submission.csv`

La olimpiada nu afisezi raspunsul pe ecran. Programul tau scrie un fisier CSV cu trei coloane, iar judgele il citeste si iti da punctele:

| subtaskID | datapointID | answer |
|---|---|---|
| 1 | 1 | pisica |
| 1 | 2 | caine |

Cate un rand pentru fiecare exemplu de test, in aceasta forma exacta (numele coloanelor conteaza, la fel si numarul de randuri). In enuntul fiecarei probleme verifici detaliile: cate subtask-uri sunt, ce fel de valori se cer in `answer`.

In [ ]:
predictii = ['pisica', 'caine', 'pisica']         # raspunsurile modelului, in ordinea exemplelor

with open('submission.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f)
    w.writerow(['subtaskID', 'datapointID', 'answer'])
    for i, raspuns in enumerate(predictii, start=1):
        w.writerow([1, i, raspuns])

with open('submission.csv', encoding='utf-8') as f:
    print(f.read())

Cu pandas iese mai scurt, dar are o capcana: `to_csv` scrie implicit si indexul randurilor, adica o **coloana in plus** pe care judgele nu o asteapta. Adauga mereu `index=False`:

In [ ]:
sub = pd.DataFrame({'subtaskID': 1, 'datapointID': range(1, 4), 'answer': predictii})
print(sub.to_csv(index=False))                    # corect: fara cale, intoarce textul, ca sa vezi ce s-ar scrie
print(sub.to_csv())                               # gresit: prima coloana (indexul) n-are ce cauta aici

Inainte sa trimiti: citeste fisierul inapoi si verifica-l (numele coloanelor, numarul de randuri, ca nu lipseste niciun `datapointID`). Cateva secunde de verificare fac diferenta dintre 0 si punctajul real.

In [ ]:
verif = pd.read_csv('submission.csv')
print(list(verif.columns))
print(len(verif), 'randuri;', 'datapointID de la', verif['datapointID'].min(), 'la', verif['datapointID'].max())

## 4. Imagini

O imagine este o **matrice de numere**. Fiecare numar e un pixel.

- imagine in **tonuri de gri**: matrice `H x W`, valori `0` (negru) ... `255` (alb);
- imagine **color**: `H x W x 3`, cele trei numere din fiecare pixel sunt intensitatile de rosu, verde si albastru (**RGB**).

Tipul e `uint8` (intreg fara semn pe 8 biti, 0-255). Il retii, pentru ca de el se leaga o capcana mai jos.

Citim imaginea cu Pillow (`PIL`) si o transformam in tablou NumPy. Urmareste cum zoom-ul coboara de la imaginea intreaga pana la 64 de numere:

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

mic = np.array(Image.open(os.path.join(ACASA, 'date', '09_mic.png')))
print(mic.shape, mic.dtype)
anim_pixeli(mic, 7, 13, 8)            # patratul 8 x 8 care incepe la randul 7, coloana 13

Acum una color. `.convert('RGB')` e un obicei bun: unele PNG-uri vin cu un canal in plus (transparenta) sau cu paleta, iar el le aduce mereu la 3 canale.

In [ ]:
forme = np.array(Image.open(os.path.join(ACASA, 'date', '09_forme.png')).convert('RGB'))
print(forme.shape, forme.dtype, forme.min(), forme.max())
print(forme[10, 10])                  # un pixel: [R, G, B]
print(forme[10, 10, 0])               # doar canalul rosu al pixelului
fig, ax = plt.subplots(1, 4, figsize=(10, 2.8))
ax[0].imshow(forme)
ax[0].set_title('RGB')
for i, nume in enumerate('RGB'):
    ax[i + 1].imshow(forme[:, :, i], cmap='gray', vmin=0, vmax=255)     # un canal = o matrice 2D
    ax[i + 1].set_title('canalul ' + nume)
for a in ax:
    a.axis('off')
plt.show()

Observa: patratul rosu e **luminos** in canalul R si intunecat in G si B. Cercul verde, invers.

Din faptul ca imaginea e un tablou, vine gratis tot ce stii de la slicing. Pe o matrice mica se vede clar:

In [ ]:
%%vezi
import numpy as np
img = np.array([[10, 20, 30, 40],
                [50, 60, 70, 80],
                [90, 100, 110, 120]])
sus_jos = img[::-1]               #: randurile in ordine inversa: oglindire pe verticala
stanga_dreapta = img[:, ::-1]     #: coloanele inversate: oglindire pe orizontala
bucata = img[0:2, 1:3]            #: decupaj: randurile 0-1, coloanele 1-2

**Capcana: decupajul nu e copie.** `img[0:2, 1:3]` e o "fereastra" catre aceeasi memorie. Daca modifici bucata, se schimba si imaginea originala. Pentru o copie adevarata: `.copy()`.

In [ ]:
%%vezi
import numpy as np
img = np.array([[10, 20, 30], [40, 50, 60]])
bucata = img[:, 0:2]
bucata[0, 0] = 999                #: bucata e o fereastra spre img: se schimba si img
copie = img[:, 0:2].copy()        #: .copy() da un tablou separat
copie[0, 1] = -1                  #: img ramane neatins

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(10, 2.8))
poze = [forme, forme[:, ::-1], forme[::-1], forme[8:28, 8:28]]
titluri = ['original', 'forme[:, ::-1]', 'forme[::-1]', 'forme[8:28, 8:28]']
for a, p, t in zip(ax, poze, titluri):
    a.imshow(p)
    a.set_title(t, fontsize=9)
    a.axis('off')
plt.show()

### Din color in tonuri de gri

Ochiul e mai sensibil la verde decat la rosu, iar la albastru cel mai putin. Formula uzuala pentru luminozitatea unui pixel:

`gri = 0.299 * R + 0.587 * G + 0.114 * B`

Un pixel, apoi o imagine 2 x 2, **din topor** (doua bucle):

In [ ]:
%%vezi
r, g, b = 205, 40, 40                        #: un pixel rosu
gri = 0.299 * r + 0.587 * g + 0.114 * b      #: verdele conteaza cel mai mult

In [ ]:
%%vezi
img = [[(255, 0, 0), (0, 255, 0)],
       [(0, 0, 255), (255, 255, 255)]]
gri = [[0, 0], [0, 0]]
for i in range(2):
    for j in range(2):
        r, g, b = img[i][j]                  #: despachetam pixelul (i, j)
        gri[i][j] = round(0.299 * r + 0.587 * g + 0.114 * b)

Aceeasi bucla pe imaginea `forme`, apoi varianta NumPy (inmultesti canalele cu ponderile si aduni pe axa 2), apoi cea din Pillow (`.convert('L')`, "L" vine de la luminance). Toate trei trebuie sa dea aproape acelasi lucru (Pillow rotunjeste la intregi):

In [ ]:
import time

def gri_bucle(img):
    h, w = img.shape[:2]
    gri = np.zeros((h, w))
    for i in range(h):
        for j in range(w):
            r, g, b = img[i, j]
            gri[i, j] = 0.299 * r + 0.587 * g + 0.114 * b
    return gri

def gri_numpy(img):
    return (img * np.array([0.299, 0.587, 0.114])).sum(axis=2)

gri = gri_bucle(forme)
verifica_egal('bucle = NumPy', gri, gri_numpy(forme))
gri_pil = np.array(Image.fromarray(forme).convert('L'))
verifica_egal('de mana = Pillow (+-1)', gri, gri_pil.astype(float), tol=1)

# si viteza, pe o bucata mai mare din fotografia din scikit-learn
from sklearn.datasets import load_sample_image
china = load_sample_image('china.jpg')[:200, :200]
t = time.perf_counter(); gri_bucle(china); t1 = time.perf_counter() - t
t = time.perf_counter(); gri_numpy(china); t2 = time.perf_counter() - t
print(f'{china.shape}: bucle {t1:.3f}s, NumPy {t2:.5f}s (~{t1 / t2:.0f}x)')

NumPy castiga de zeci sau sute de ori din acelasi motiv ca in orice alt loc: bucla ruleaza in C, pe memorie continua, in loc sa treaca prin interpretorul Python pentru fiecare din cele 40.000 de pixeli.

Ramai cu ideea, nu cu functia: **o operatie pe toata imaginea deodata e mai rapida si mai scurta decat bucla**. `.convert('L')` e cel mai scurt, dar acum stii ce face.

In [ ]:
gri_u8 = np.round(gri).astype(np.uint8)          # inapoi la intregi 0-255
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(forme)
ax[1].imshow(gri_u8, cmap='gray', vmin=0, vmax=255)      # vmin/vmax: altfel matplotlib "intinde" contrastul
for a in ax:
    a.axis('off')
plt.show()

Image.fromarray(gri_u8).save('gri.png')                  # salvare: din tablou uint8 inapoi in fisier
print(np.array(Image.open('gri.png')).shape)

### Luminozitate: capcana `uint8`

Sa luminam o imagine inseamna sa adunam un numar la fiecare pixel. Numai ca un `uint8` nu poate trece de 255: cand depaseste, **se rostogoleste** de la 0 (ca odometrul).

**Prezice:** ce iese la `250 + 10` daca `250` e un `uint8`?

<details><summary>Raspuns</summary>

`4`. `260` nu incape in 0-255, deci ramane `260 - 256 = 4`. Un pixel aproape alb devine aproape negru. Pe tablouri NumPy nu apare nicio eroare, doar imaginea iese cu pete ciudate.
</details>

In [ ]:
%%vezi
import numpy as np
a = np.array([100, 200, 250], dtype=np.uint8)
gresit = a + 10                                             #: 250 + 10 nu incape in uint8: se rostogoleste la 4
corect = np.clip(a.astype(int) + 10, 0, 255).astype(np.uint8)   #: trecem la int, taiem la 255, ne intoarcem la uint8

In [ ]:
gresit = gri_u8 + 60                                                    # uint8 + 60: pixelii albi ies negri
corect = np.clip(gri_u8.astype(int) + 60, 0, 255).astype(np.uint8)      # clip taie la 255

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, p, t in zip(ax, [gri_u8, gresit, corect], ['original', '+ 60 gresit', '+ 60 cu clip']):
    a.imshow(p, cmap='gray', vmin=0, vmax=255)
    a.set_title(t)
    a.axis('off')
plt.show()

(Pe un scalar NumPy, `np.uint8(250) + 10` da acelasi `4`, dar cu un avertisment, in functie de versiune. Bug-ul cel mai des apare pe tablouri, si acolo e mut.)

O mentiune, doar ca sa nu te ia prin surprindere: biblioteca **OpenCV** (`cv2`) incarca imaginile color in ordinea **BGR**, nu RGB. Daca o afisezi cu matplotlib fara sa inversezi canalele, rosul si albastrul sunt schimbate. Pillow si scikit-learn folosesc RGB.

### Prag alb / negru si histograma

**Pragul** transforma o imagine gri in alb-negru: pixelul e alb daca depaseste un prag, altfel negru. Comparatia se face pe toata matricea deodata si da o matrice de `True`/`False`:

In [ ]:
%%vezi
import numpy as np
g = np.array([[10, 200, 130], [90, 250, 40]])
masca = g > 128                   #: comparatia se face pe fiecare pixel; rezulta True/False
albe = masca.sum()                #: True conteaza 1, deci suma numara pixelii albi
bw = masca * 255                  #: True -> 255, False -> 0: imagine alb-negru

Care prag e bun? Depinde de imagine. Animatia de mai jos e facuta cu `gri_u8`, matricea pe care ai calculat-o tu mai sus. Uita-te la histograma din dreapta (cati pixeli au fiecare valoare 0-255): pragul potrivit e in "valea" dintre cele doua grupuri de valori.

In [ ]:
anim_prag(gri_u8)

Histograma se calculeaza usor de mana: un contor pentru fiecare valoare posibila.

In [ ]:
%%vezi
pixeli = [3, 0, 3, 1, 3]
h = [0, 0, 0, 0]                  #: cate un contor pentru fiecare valoare posibila 0..3
for v in pixeli:
    h[v] += 1                     #: valoarea pixelului e chiar pozitia contorului

In [ ]:
h = [0] * 256
for v in gri_u8.ravel():                      # ravel(): matricea desfacuta intr-un sir de pixeli
    h[v] += 1
h_np = np.bincount(gri_u8.ravel(), minlength=256)         # aceeasi idee, scrisa in C
verifica_egal('histograma de mana = bincount', np.array(h), h_np)

plt.bar(range(256), h_np, width=1.0, color='gray')
plt.xlabel('valoarea pixelului')
plt.ylabel('cati pixeli')
plt.show()

## 5. Capcane

**Capcana 1: `'w'` sterge.** Ai vazut-o la inceput. Daca vrei sa adaugi la un fisier (un jurnal, rezultate pe parcurs), foloseste `'a'`.

In [ ]:
for _ in range(2):
    with open('jurnal.txt', 'w') as f:
        f.write('pornit\n')
with open('jurnal.txt') as f:
    print(len(f.readlines()), 'linie (dupa doua rulari cu w)')

**Capcana 2: CSV da doar text.** Un `'9'` citit din fisier nu e numarul 9. Adunarea lipeste, iar sortarea e alfabetica:

In [ ]:
%%vezi
valori = ['10', '9', '8']
lipite = valori[0] + valori[1]            #: + intre texte inseamna lipire
sortate = sorted(valori)                  #: '10' vine inaintea lui '8': comparatie caracter cu caracter
bune = sorted(int(v) for v in valori)     #: dupa conversie, ordinea e cea numerica

**Capcana 3: `\n` la capatul liniei.** Linia citita din fisier nu e egala cu textul din spatele ei. Compara doar dupa `strip()`:

In [ ]:
%%vezi
with open('note.txt', encoding='utf-8') as f:
    prima = f.readline()                  #: 'Ana 9\n'
egal = prima == 'Ana 9'                   #: False: \n conteaza
egal_strip = prima.strip() == 'Ana 9'     #: True

**Capcana 4: imaginea nu are mereu 3 canale.** Un PNG cu transparenta se incarca cu 4 canale (RGBA), si formula gri cu `img[..., 2]` merge in continuare, dar ai citit alt lucru decat credeai. Verifica `shape` si converteste:

In [ ]:
rgba = Image.new('RGBA', (2, 2), (255, 0, 0, 128))
print(np.array(rgba).shape)                       # (2, 2, 4): al patrulea canal e transparenta
print(np.array(rgba.convert('RGB')).shape)        # (2, 2, 3)

## 6. Cum citesti eroarea

Cu fisiere apar erori pe care nu le-ai vazut pana acum. Le vezi in animatii, cu linia rosie exact la cauza.

**`FileNotFoundError`** - calea e gresita, fisierul nu e in folderul curent, sau ai uitat sa-l aduci. Verifica `os.getcwd()` si `os.path.exists(cale)`.

In [ ]:
%%vezi
with open('nu_exista.txt') as f:
    text = f.read()

**`UnicodeDecodeError`** - ai deschis fisierul cu alt encoding decat cel in care a fost scris (aici, `ascii` pentru un fisier cu litere romanesti). Solutia: `encoding='utf-8'`.

In [ ]:
%%vezi
with open('ro.txt', encoding='ascii') as f:
    text = f.read()

**`ValueError: I/O operation on closed file`** - ai folosit fisierul dupa ce s-a terminat blocul `with` (care l-a inchis).

In [ ]:
%%vezi
with open('note.txt', encoding='utf-8') as f:
    prima = f.readline()
continut = f.read()

**`ValueError: could not convert string to float: ''`** - o celula goala din CSV. Prinzi eroarea sau verifici `if celula:` inainte de conversie.

In [ ]:
%%vezi
celula = ''
nota = float(celula)

**`KeyError: 'Nota'`** - numele coloanei din `DictReader` nu se potriveste (mare / mica conteaza, la fel si spatiile din antet).

In [ ]:
%%vezi
import csv
import io
f = io.StringIO('nume,nota\nAna,9\n')
for r in csv.DictReader(f):
    valoare = r['Nota']

**`ValueError: operands could not be broadcast together`** - formele nu se potrivesc: aici ai inmultit o imagine `(H, W, 3)` cu ponderi pentru doar doua canale. Verifica mereu `.shape` inainte de operatii pe imagini.

In [ ]:
%%vezi
import numpy as np
img = np.array([[[255, 0, 0], [0, 255, 0]], [[0, 0, 255], [9, 9, 9]]])
gri = (img * np.array([0.299, 0.587])).sum(axis=2)

| eroare | de obicei inseamna |
|---|---|
| `FileNotFoundError` | cale gresita sau fisier neadus |
| `UnicodeDecodeError` | encoding gresit: foloseste `utf-8` |
| `ValueError: I/O operation on closed file` | ai folosit `f` dupa blocul `with` |
| `ValueError: could not convert string to float` | celula goala sau text in coloana numerica |
| `KeyError` la `DictReader` | numele coloanei nu se potriveste cu antetul |
| `ValueError: operands could not be broadcast` | formele (`shape`) tablourilor nu se potrivesc |
| `IndexError` la o linie | linie goala (de obicei ultima) sau rand prea scurt |

## Exercitiile de clasa

Verificarile de mai jos creeaza singure fisiere temporare (functia `pe_fisier`), apeleaza functia ta pe ele si le sterg. Tu scrii doar functia care primeste **calea** fisierului. Celula urmatoare defineste `pe_fisier`: ruleaz-o o data.

In [ ]:
import os, tempfile

def pe_fisier(f, continut, *alte, nume='a.txt'):
    # scrie 'continut' intr-un fisier temporar, apeleaza f(cale, *alte), sterge fisierul
    with tempfile.TemporaryDirectory() as d:
        cale = os.path.join(d, nume)
        with open(cale, 'w', encoding='utf-8', newline='') as g:
            g.write(continut)
        return f(cale, *alte)

import numpy as np

**Exercitiul 1.** `linii_cuvinte(cale)`: intoarce perechea `(numar_de_linii, numar_de_cuvinte)` a fisierului. O linie goala conteaza ca linie; cuvintele sunt separate prin spatii.

In [ ]:
def linii_cuvinte(cale):
    pass  # TODO

In [ ]:
verifica('linii_cuvinte', lambda text: pe_fisier(linii_cuvinte, text),
         [(('ana are mere\nsi pere\n',), (2, 5)), (('a\n\nb',), (3, 2))],
         ascunse=[(('',), (0, 0)), (('un cuvant',), (1, 2)), (('\n\n',), (2, 0)), (('  a   b  \nc\n',), (2, 3))],
         indiciu='for linie in f: numeri linia, apoi len(linie.split()) cuvinte')

**Exercitiul 2.** `suma_din_fisier(cale)`: fisierul are cate un numar intreg pe linie (pot fi si linii goale, pe care le ignori). Intoarce suma lor.

In [ ]:
def suma_din_fisier(cale):
    pass  # TODO

In [ ]:
verifica('suma_din_fisier', lambda text: pe_fisier(suma_din_fisier, text),
         [(('1\n2\n3\n',), 6), (('10\n\n-4\n',), 6)],
         ascunse=[(('',), 0), ((' 5 \n7',), 12), (('-1\n-2\n',), -3), (('1000000\n2000000\n',), 3000000)],
         indiciu='sari peste liniile goale: if linie.strip(): ...; int() accepta si \\n la capat')

**Exercitiul 3.** `media_coloana(cale, coloana)`: fisierul e CSV cu antet. Intoarce media valorilor din coloana `coloana`, **sarind celulele care nu sunt numere** (goale, text, sau randuri prea scurte). Daca nu exista nicio valoare buna, intoarce `None`. Foloseste `csv.DictReader` (numele pot contine virgule, intre ghilimele).

In [ ]:
import csv


def media_coloana(cale, coloana):
    pass  # TODO

In [ ]:
verifica('media_coloana', lambda text, col: pe_fisier(media_coloana, text, col, nume='a.csv'),
         [(('nume,nota\nAna,9\nDan,\nEva,7\n', 'nota'), 8.0),
          (('nume,nota\n"Popescu, Ion",10\nAna,8\n', 'nota'), 9.0)],
         ascunse=[(('nume,nota\nAna,absent\nDan,6\n', 'nota'), 6.0), (('nume,nota\nAna,\nDan,\n', 'nota'), None),
                  (('nume,nota,clasa\nAna,9,10\nDan\nEva,7,11\n', 'nota'), 8.0), (('a,b\n1,2\n3,4\n', 'b'), 3.0),
                  (('a,b\n1.5,2\n2.5,4\n', 'a'), 2.0)],
         indiciu='try: float(r[coloana]) ... except (ValueError, TypeError): o celula lipsa poate fi \'\' sau None')

**Exercitiul 4.** `scrie_submission(cale, subtask, raspunsuri)`: scrie un CSV in formatul judge-ului. Antetul `subtaskID,datapointID,answer`, apoi cate un rand pentru fiecare raspuns din lista: `subtask`, numarul de ordine al exemplului (de la 1) si raspunsul. Foloseste `csv.writer`.

In [ ]:
import csv


def scrie_submission(cale, subtask, raspunsuri):
    pass  # TODO

In [ ]:
def t_scrie(subtask, raspunsuri):
    with tempfile.TemporaryDirectory() as d:
        cale = os.path.join(d, 's.csv')
        scrie_submission(cale, subtask, raspunsuri)
        with open(cale, encoding='utf-8') as f:
            return f.read().splitlines()

H = 'subtaskID,datapointID,answer'
verifica('scrie_submission', t_scrie,
         [((1, ['a', 'b']), [H, '1,1,a', '1,2,b']), ((2, [0, 1, 1]), [H, '2,1,0', '2,2,1', '2,3,1'])],
         ascunse=[((1, []), [H]), ((3, ['x,y']), [H, '3,1,"x,y"']), ((1, [7]), [H, '1,1,7'])],
         indiciu='csv.writer, open(..., \'w\', newline=\'\'), enumerate(raspunsuri, start=1); un raspuns cu virgula are nevoie de ghilimele')

**Exercitiul 5.** `gri_de_mana(img)`: primeste o imagine color (`H x W x 3`) si intoarce matricea `H x W` de tip `float` cu formula `0.299 * R + 0.587 * G + 0.114 * B`, **fara rotunjire**. Poti face bucle sau NumPy.

In [ ]:
def gri_de_mana(img):
    pass  # TODO

In [ ]:
rgb_mic = np.array([[[255, 0, 0], [0, 255, 0]], [[0, 0, 255], [255, 255, 255]]], dtype=np.uint8)
asteptat = np.array([[0.299 * 255, 0.587 * 255], [0.114 * 255, 255.0]])
verifica('gri_de_mana', gri_de_mana, [((rgb_mic,), asteptat)],
         referinta=lambda im: (im * np.array([0.299, 0.587, 0.114])).sum(axis=2),
         generator=lambda: (np.random.randint(0, 256, (np.random.randint(1, 6), np.random.randint(1, 6), 3)).astype(np.uint8),),
         indiciu='r, g, b = img[i, j] si apoi formula; rezultatul e o matrice de float H x W')

# aceeasi verificare, dar pe imaginea reala si fata de Pillow (care rotunjeste, de aceea tol=1)
forme_ex = np.array(Image.open(os.path.join(ACASA, 'date', '09_forme.png')).convert('RGB'))
gri_pil_ex = np.array(Image.fromarray(forme_ex).convert('L')).astype(float)
verifica('gri_de_mana vs Pillow', gri_de_mana, [((forme_ex,), gri_pil_ex)], tol=1)

**Exercitiul 6.** `oglinda_si_centru(img)`: intoarce o pereche `(oglinda, centru)`. `oglinda` e imaginea oglindita stanga-dreapta. `centru` e bucata din mijloc: randurile de la `H // 4` la `3 * H // 4` (exclusiv), coloanele de la `W // 4` la `3 * W // 4` (exclusiv). Merge la fel pentru imagini gri si color: doar slicing, fara bucle.

In [ ]:
def oglinda_si_centru(img):
    pass  # TODO

In [ ]:
def ref_oglinda(img):
    h, w = img.shape[:2]
    return img[:, ::-1].copy(), img[h // 4:3 * h // 4, w // 4:3 * w // 4].copy()

def gen_oglinda():
    forma = (np.random.randint(4, 11), np.random.randint(4, 11)) + ((3,) if np.random.rand() < 0.5 else ())
    return (np.random.randint(0, 256, forma).astype(np.uint8),)

mica = np.arange(16, dtype=np.uint8).reshape(4, 4)
verifica('oglinda_si_centru', oglinda_si_centru, [((mica,), (mica[:, ::-1], mica[1:3, 1:3]))],
         referinta=ref_oglinda, generator=gen_oglinda,
         indiciu='oglinda: img[:, ::-1]; centru: img[h // 4:3 * h // 4, w // 4:3 * w // 4]')

**Exercitiul 7.** `pixeli_peste(img, prag)`: cati pixeli dintr-o imagine gri au valoarea **strict mai mare** decat `prag`. Fara bucle: `(img > prag).sum()`.

In [ ]:
def pixeli_peste(img, prag):
    pass  # TODO

In [ ]:
g1 = np.array([[10, 200, 130], [90, 250, 40]], dtype=np.uint8)
verifica('pixeli_peste', pixeli_peste, [((g1, 128), 3), ((g1, 250), 0), ((g1, 0), 6)],
         ascunse=[((g1, 255), 0), ((g1, 39), 5), ((np.zeros((3, 3), dtype=np.uint8), 0), 0)],
         referinta=lambda im, p: int(np.sum(im.ravel() > p)),
         generator=lambda: (np.random.randint(0, 256, (5, 7)).astype(np.uint8), np.random.randint(0, 256)),
         indiciu='comparatia (img > prag) da o matrice de True/False; sum() le numara')

## Recapitulare

- **Text:** `with open(cale, mod, encoding='utf-8') as f`. Moduri: `r` (citeste), `w` (scrie si **sterge**), `a` (adauga). Bucla `for linie in f` + `.strip()` / `.split()`. Nu `read()` pe fisiere mari.
- `write` nu pune `\n`, `print(..., file=f)` il pune. Linia citita are `\n` la capat.
- Fisier lipsa: `os.path.exists`, `try/except FileNotFoundError`.
- **CSV:** `csv.reader` / `csv.DictReader` / `csv.writer`, cu `newline=''`. Totul vine ca **text**: conversie cu `try/except`. Pandas (`read_csv`) face totul intr-o linie, mai rapid.
- **submission.csv** = `subtaskID,datapointID,answer`, cate un rand pe exemplu. Cu pandas: `to_csv(index=False)`.
- **Imagini:** matrice de numere (`H x W` gri, `H x W x 3` RGB), `uint8`. `Image.open` -> `np.array`; oglindire si decupaj cu slicing (decupajul e view, nu copie); gri = `0.299 R + 0.587 G + 0.114 B`; prag = comparatie pe toata matricea; histograma = contor pe valoare.
- Capcana `uint8`: `250 + 10` face `4`. Trece la `int`, aplica `np.clip(..., 0, 255)`, intoarce-te la `uint8`.
- OpenCV foloseste BGR, nu RGB.

Tema (notebook-ul urmator): mai multe exercitii pe text si CSV, un program in format concurs, un jurnal de reparat, iluminare cu clip, histograma si un blur 3 x 3 pe care il scrii singur.